In [1]:
"""Compare persistence baseline, saved unified model, and per-station models.
This reuses test_predictions.csv from the existing unified-model run. It does not
retrain the original 10 unified candidates.
"""
import os, warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


BASE_DIR = (r"C:\Users\hanus\Downloads\beijing+multi+site+air+quality+data"
             r"\PRSA2017_Data_20130301-20170228\PRSA_Data_20130301-20170228")
OUTPUT_DIR = os.getcwd()  # place beside test_predictions.csv, or change this
STATIONS = ["Aotizhongxin", "Changping", "Dingling", "Dongsi", "Guanyuan",
            "Gucheng", "Huairou", "Nongzhanguan", "Shunyi", "Tiantan",
            "Wanliu", "Wanshouxigong"]
SEED = 42
TRAIN_FRAC, VAL_FRAC = 0.70, 0.10
MODEL_PARAMS = dict(max_iter=300, learning_rate=0.05, random_state=SEED)
RESULT_DIR = os.path.join(OUTPUT_DIR, "baseline_vs_station_models")
GRAPH_DIR = os.path.join(RESULT_DIR, "graphs")
os.makedirs(GRAPH_DIR, exist_ok=True)



def scores(y, p):
    return {"RMSE": float(np.sqrt(mean_squared_error(y, p))),
            "MAE": float(mean_absolute_error(y, p)),
            "R2": float(r2_score(y, p)),
            "Samples": int(len(y))}



def savefig(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(GRAPH_DIR, name), dpi=220, bbox_inches="tight")
    plt.close(fig)



# Load and reproduce the preprocessing/feature engineering used in the unified script.
frames = []
for station in STATIONS:
    file_path = os.path.join(BASE_DIR, f"PRSA_Data_{station}_20130301-20170228.csv")
    if not os.path.isfile(file_path):
        print(f"Missing station file: {file_path}")
        continue
    d = pd.read_csv(file_path)
    d["station"] = station
    frames.append(d)
if len(frames) != len(STATIONS):
    raise FileNotFoundError("Not all 12 station files were found. Check BASE_DIR.")
df = pd.concat(frames, ignore_index=True)
df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]], errors="coerce")
df = df.dropna(subset=["datetime"]).sort_values(["station", "datetime"]).reset_index(drop=True)
df["station_encoded"] = LabelEncoder().fit_transform(df["station"])
num_cols = [c for c in ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"] if c in df]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for c in num_cols:
    if c != "PM2.5":
        df[c] = df.groupby("station")[c].ffill()
if "wd" in df:
    df["wd"] = df.groupby("station")["wd"].ffill().fillna("Unknown")
    df["wd_encoded"] = LabelEncoder().fit_transform(df["wd"].astype(str))
else:
    df["wd_encoded"] = 0
# Time features
df["month"] = df["datetime"].dt.month
df["hour"] = df["datetime"].dt.hour
df["dayofweek"] = df["datetime"].dt.dayofweek
df["dayofyear"] = df["datetime"].dt.dayofyear
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
df["month_cos"] = np.cos(2*np.pi*df["month"]/12)
gpm = df.groupby("station")["PM2.5"]
for lag in [1, 3, 6, 12, 24]:
    df[f"PM2.5_lag_{lag}"] = gpm.shift(lag)
    if "TEMP" in df:
        df[f"TEMP_lag_{lag}"] = df.groupby("station")["TEMP"].shift(lag)
prev_pm = gpm.shift(1)
by_station = prev_pm.groupby(df["station"])
for window in [3, 6, 12, 24]:
    df[f"PM2.5_rolling_mean_{window}"] = by_station.rolling(window, min_periods=1).mean().reset_index(level=0, drop=True)
    df[f"PM2.5_rolling_std_{window}"] = by_station.rolling(window, min_periods=2).std().reset_index(level=0, drop=True)
if {"TEMP", "PRES"}.issubset(df.columns):
    df["TEMP_PRES_interaction"] = df["TEMP"] * df["PRES"]
if {"TEMP", "DEWP"}.issubset(df.columns):
    df["TEMP_DEWP_interaction"] = df["TEMP"] * df["DEWP"]
if {"NO2", "SO2"}.issubset(df.columns):
    df["NO2_SO2_interaction"] = df["NO2"] * df["SO2"]
df = df.dropna(subset=["PM2.5"]).copy()
exclude = {"datetime", "station", "year", "day", "PM2.5", "wd", "No"}
feature_cols = [c for c in df.columns if c not in exclude]

# Use identical global date cutoffs as the unified experiment.
dates = np.sort(df["datetime"].unique())
nd = len(dates)
train_i = max(1, min(int(nd*TRAIN_FRAC), nd-2))
val_i = max(train_i+1, min(int(nd*(TRAIN_FRAC+VAL_FRAC)), nd-1))
train_cut, val_cut = pd.Timestamp(dates[train_i-1]), pd.Timestamp(dates[val_i-1])
trainval = df[df["datetime"] <= val_cut].copy()
test = df[df["datetime"] > val_cut].copy()

# Load existing saved unified-model test predictions and align by station/time.
unified_path = os.path.join(OUTPUT_DIR, "test_predictions.csv")
if not os.path.exists(unified_path):
    raise FileNotFoundError("test_predictions.csv not found. Run your unified model script first.")
unified = pd.read_csv(unified_path)
unified["datetime"] = pd.to_datetime(unified["datetime"])
merged = test[["station", "datetime", "PM2.5", "PM2.5_lag_1"]].merge(
    unified[["station", "datetime", "Predicted_PM2.5"]], on=["station", "datetime"], how="inner")
merged = merged.rename(columns={"PM2.5": "Actual_PM2.5", "Predicted_PM2.5": "Unified_Prediction", "PM2.5_lag_1": "Persistence_Prediction"})
merged = merged.dropna(subset=["Actual_PM2.5", "Unified_Prediction", "Persistence_Prediction"]).copy()
print(f"Common comparison samples (all 3 methods): {len(merged):,}")

# Train a separate model per station on that station's train+validation period.
station_preds = []
for station in STATIONS:
    tr = trainval[trainval["station"] == station].sort_values("datetime")
    te = test[test["station"] == station].sort_values("datetime")
    if tr.empty or te.empty:
        print(f"Skipping {station}: empty train/test")
        continue
    ytr = tr["PM2.5"].to_numpy()
    yte = te["PM2.5"].to_numpy()
    imputer = SimpleImputer(strategy="median")
    scaler = RobustScaler()
    xtr = scaler.fit_transform(imputer.fit_transform(tr[feature_cols]))
    xte = scaler.transform(imputer.transform(te[feature_cols]))
    model = HistGradientBoostingRegressor(**MODEL_PARAMS)
    t0 = time.time()
    model.fit(xtr, ytr)
    p = model.predict(xte)
    print(f"Station model {station}: {len(tr):,} train / {len(te):,} test | {time.time()-t0:.1f}s")
    station_preds.append(pd.DataFrame({"station": station, "datetime": te["datetime"].values,
                                     "Station_Model_Prediction": p, "Actual_PM2.5": yte}))
if not station_preds:
    raise RuntimeError("No station models were trained.")
station_predictions = pd.concat(station_preds, ignore_index=True)
comparison = merged.merge(station_predictions[["station", "datetime", "Station_Model_Prediction"]], on=["station", "datetime"], how="inner")
comparison = comparison.dropna(subset=["Actual_PM2.5", "Unified_Prediction", "Persistence_Prediction", "Station_Model_Prediction"])

# Overall results on the same rows for a fair comparison.
overall_rows = []
for name, col in [("Previous-hour baseline", "Persistence_Prediction"),
                   ("Unified model (saved predictions)", "Unified_Prediction"),
                   ("Individual station models", "Station_Model_Prediction")]:
    overall_rows.append({"Method": name, **scores(comparison["Actual_PM2.5"], comparison[col])})
overall = pd.DataFrame(overall_rows).sort_values("RMSE")
overall.to_csv(os.path.join(RESULT_DIR, "overall_method_comparison.csv"), index=False)
print("\nOVERALL COMPARISON - same test rows")
print(overall.to_string(index=False))

# Per-station score table
rows = []
for station, part in comparison.groupby("station"):
    for method, col in [("Previous-hour baseline", "Persistence_Prediction"),
                        ("Unified model", "Unified_Prediction"),
                        ("Individual station model", "Station_Model_Prediction")]:
        rows.append({"Station": station, "Method": method, **scores(part["Actual_PM2.5"], part[col])})
per_station = pd.DataFrame(rows)
per_station.to_csv(os.path.join(RESULT_DIR, "station_method_comparison.csv"), index=False)
print("\nPER-STATION COMPARISON")
print(per_station.to_string(index=False))
comparison.to_csv(os.path.join(RESULT_DIR, "comparison_test_predictions.csv"), index=False)

# Plot 1: overall RMSE/MAE comparison
fig, ax = plt.subplots(figsize=(10, 6))
plot = overall.set_index("Method")[["RMSE", "MAE"]]
plot.plot(kind="bar", ax=ax)
ax.set(ylabel="Error (µg/m³)", title="Baseline vs Unified vs Individual Models")
ax.tick_params(axis="x", rotation=20); ax.grid(axis="y", alpha=.2)
plt.tight_layout(); savefig(fig, "01_overall_rmse_mae_comparison.png")

# Plot 2: R2 comparison
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(overall["Method"], overall["R2"])
ax.set(ylabel="R²", title="Overall R² Comparison")
ax.tick_params(axis="x", rotation=20); ax.grid(axis="y", alpha=.2)
plt.tight_layout(); savefig(fig, "02_overall_r2_comparison.png")

# Plot 3: station x method RMSE
pivot_rmse = per_station.pivot(index="Station", columns="Method", values="RMSE")
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(pivot_rmse, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax)
ax.set(title="Station-wise RMSE: Baseline, Unified, Individual", xlabel="Method", ylabel="Station")
plt.tight_layout(); savefig(fig, "03_station_method_rmse_heatmap.png")

# Plot 4: station-wise winner counts
winner = per_station.loc[per_station.groupby("Station")["RMSE"].idxmin(), ["Station", "Method"]]
win_counts = winner["Method"].value_counts()
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(win_counts.index, win_counts.values)
ax.set(ylabel="Number of stations", title="Which Method Wins at Each Station (lowest RMSE)")
ax.tick_params(axis="x", rotation=20); ax.grid(axis="y", alpha=.2)
plt.tight_layout(); savefig(fig, "04_station_winner_counts.png")

# Plot 5: per-station RMSE grouped bars
fig, ax = plt.subplots(figsize=(14, 7))
pivot_rmse.plot(kind="bar", ax=ax)
ax.set(ylabel="RMSE (µg/m³)", title="Per-station RMSE: Three Methods")
ax.tick_params(axis="x", rotation=40); ax.grid(axis="y", alpha=.2)
plt.tight_layout(); savefig(fig, "05_per_station_rmse_grouped.png")

# Plot 6: unified-vs-individual RMSE difference; positive means individual is worse.
diff = (pivot_rmse["Individual station model"] - pivot_rmse["Unified model"]).sort_values()
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(diff.index, diff.values)
ax.axvline(0, color="black", linestyle="--")
ax.set(xlabel="Individual RMSE minus Unified RMSE (µg/m³)", title="Where Unified Model Performs Better (positive = unified better)")
ax.grid(axis="x", alpha=.2)
plt.tight_layout(); savefig(fig, "06_unified_vs_individual_rmse_difference.png")

print("\nExperiment completed. Results and graphs saved to:", RESULT_DIR)


Common comparison samples (all 3 methods): 81,949


  File "C:\Users\hanus\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\hanus\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\hanus\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 1036, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                        start_new_session, process_group)
                

Station model Aotizhongxin: 27,223 train / 6,916 test | 5.2s
Station model Changping: 27,351 train / 6,939 test | 4.7s
Station model Dingling: 27,494 train / 6,791 test | 4.0s
Station model Dongsi: 27,603 train / 6,711 test | 3.9s
Station model Guanyuan: 27,582 train / 6,866 test | 6.9s
Station model Gucheng: 27,514 train / 6,904 test | 6.4s
Station model Huairou: 27,178 train / 6,933 test | 3.0s
Station model Nongzhanguan: 27,513 train / 6,923 test | 4.3s
Station model Shunyi: 27,314 train / 6,837 test | 4.6s
Station model Tiantan: 27,457 train / 6,930 test | 5.0s
Station model Wanliu: 27,743 train / 6,939 test | 4.6s
Station model Wanshouxigong: 27,506 train / 6,862 test | 5.2s

OVERALL COMPARISON - same test rows
                           Method      RMSE       MAE       R2  Samples
Unified model (saved predictions) 11.617268  6.679216 0.980646    81949
        Individual station models 12.641941  7.035757 0.977081    81949
           Previous-hour baseline 19.567681 10.092143 0.94